# 02 不动点迭代与加速

上一节的二分法把根始终限制在变号区间内。本节转向开方法：把方程 $f(x)=0$ 改写为

$$
x=g(x),
$$

然后从初值 $x_0$ 出发构造

$$
x_{k+1}=g(x_k).
$$

这种方法的优点是形式简单、只需函数值；缺点是收敛完全依赖改写方式和初值。若 $\alpha=g(\alpha)$，且在根附近 $|g'(x)|\le q<1$，则迭代在局部是压缩映射，误差近似满足

$$
e_{k+1}\approx g'(\alpha)e_k.
$$

因此当 $|g'(\alpha)|$ 接近 1 时，简单不动点迭代会很慢；Aitken 和 Steffensen 方法正是为加速这类线性收敛序列而设计。


In [ ]:
import math
import pathlib
import sys

import numpy as np

ROOT = pathlib.Path.cwd()
while ROOT.name != "py-sc" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import aitken_delta_squared, fixed_point_iteration, steffensen_method


## 教学式不动点迭代

对 $x_{k+1}=g(x_k)$，最直接的停止准则是相邻两次迭代的变化量

$$
|x_{k+1}-x_k|\le \varepsilon\max(1, |x_{k+1}|).
$$

实际实现还应检查固定点残差 $|g(x_k)-x_k|$。下面的教学版本保留完整历史，用于观察线性收敛速度。


In [ ]:
def teaching_fixed_point(g, x0, tolerance=1e-10, max_iterations=100):
    history = [float(x0)]
    x_old = float(x0)
    for k in range(1, max_iterations + 1):
        x_new = float(g(x_old))
        history.append(x_new)
        step = abs(x_new - x_old)
        residual = abs(g(x_new) - x_new)
        if step <= tolerance * max(1.0, abs(x_new)) or residual <= tolerance:
            return x_new, k, True, np.array(history)
        x_old = x_new
    return x_old, max_iterations, False, np.array(history)

root, iterations, converged, history = teaching_fixed_point(math.cos, 0.5, tolerance=1e-12)
official = fixed_point_iteration(math.cos, 0.5, tolerance=1e-12)

print(f"teaching root = {root:.15f}, iterations = {iterations}, converged = {converged}")
print(f"official root = {official.root:.15f}, residual = {official.residual:.3e}")
print("first iterates:", np.array2string(history[:6], precision=12))


## Aitken $\Delta^2$ 加速

若序列近似满足

$$
x_k = \alpha + Cq^k,\qquad |q|<1,
$$

则可以用三个连续值消去主误差项：

$$
\widehat{x}_k
= x_k - \frac{(x_{k+1}-x_k)^2}{x_{k+2}-2x_{k+1}+x_k}.
$$

这个公式对严格的线性误差模型能一次得到极限；但当分母很小或数据受舍入误差污染时会变得不稳定。


In [ ]:
exact = 2.0
linear_sequence = np.array([exact + 0.8**k for k in range(8)], dtype=float)
accelerated_linear = aitken_delta_squared(linear_sequence)
accelerated_cos = aitken_delta_squared(history[:10])

print("linear sequence:", np.array2string(linear_sequence[:5], precision=10))
print("Aitken linear:", np.array2string(accelerated_linear[:3], precision=12))
print("Aitken on cos fixed-point:", np.array2string(accelerated_cos[:5], precision=12))


## Steffensen 方法

Aitken 加速需要已有序列。Steffensen 方法把它直接嵌入不动点迭代：从 $x_k$ 开始先算

$$
y_k=g(x_k),\qquad z_k=g(y_k),
$$

再作

$$
x_{k+1}=x_k-\frac{(y_k-x_k)^2}{z_k-2y_k+x_k}.
$$

它不显式使用导数，却常表现出接近 Newton 法的速度。代价是每步需要两次函数调用，并且 Aitken 分母接近零时必须停止或改用保护策略。


In [ ]:
def teaching_steffensen(g, x0, tolerance=1e-10, max_iterations=50):
    history = [float(x0)]
    x_old = float(x0)
    for k in range(1, max_iterations + 1):
        if abs(g(x_old) - x_old) <= tolerance:
            return x_old, k - 1, True, np.array(history)
        y = float(g(x_old))
        z = float(g(y))
        denominator = z - 2.0 * y + x_old
        if abs(denominator) <= 1e-14:
            raise ValueError("Aitken denominator is too small")
        x_new = x_old - (y - x_old) ** 2 / denominator
        history.append(x_new)
        if abs(x_new - x_old) <= tolerance * max(1.0, abs(x_new)):
            return x_new, k, True, np.array(history)
        x_old = x_new
    return x_old, max_iterations, False, np.array(history)

plain = fixed_point_iteration(math.cos, 0.5, tolerance=1e-12)
steff = steffensen_method(math.cos, 0.5, tolerance=1e-12)
teaching = teaching_steffensen(math.cos, 0.5, tolerance=1e-12)

print(f"plain fixed point iterations = {plain.iterations}, root = {plain.root:.15f}")
print(f"Steffensen iterations = {steff.iterations}, root = {steff.root:.15f}")
print(f"teaching Steffensen iterations = {teaching[1]}, root = {teaching[0]:.15f}")


## 失败情形和方法选择

不动点迭代不是“把方程移项即可”的机械步骤。同一个 $f(x)=0$ 可以有很多 $g(x)$，而不同改写的 $g'(\alpha)$ 差异很大：

* 若 $|g'(\alpha)|<1$，根附近通常局部收敛；
* 若 $|g'(\alpha)|>1$，初值即使很接近根也可能离开；
* 若 Aitken 分母接近零，加速公式会放大舍入误差；
* 若迭代历史出现震荡或发散，应退回括区间方法，或改用带保护的 Newton 类方法。


In [ ]:
def divergent_g(x):
    return 2.0 * x

slow = fixed_point_iteration(lambda x: 1.0 + 0.95 * (x - 1.0), 2.0, tolerance=1e-10, max_iterations=20)
divergent = fixed_point_iteration(divergent_g, 0.1, tolerance=1e-10, max_iterations=6)

print("slow contraction converged:", slow.converged, "iterations:", slow.iterations, "last residual:", f"{slow.residual:.3e}")
print("divergent fixed point converged:", divergent.converged, "history:", np.array2string(divergent.history, precision=6))


## 本节小结

不动点迭代的核心是选择合适的 $g$。局部收敛条件可以用 $|g'(\alpha)|<1$ 理解，误差通常线性下降。Aitken $\Delta^2$ 公式利用三个连续迭代值估计并消去主线性误差；Steffensen 方法则把这一加速过程直接嵌入迭代，常能大幅减少迭代次数。加速并不等于全局可靠：当初值不合适、改写不收敛或分母退化时，应回到上一节的括区间方法，或使用后续带导数和阻尼保护的 Newton 类方法。
